# Biofilter — Report: Variant Modeling

Map an input list of variants (rsID, chr:pos, or chr:pos:ref:alt) to biologically connected
**variant×variant pairs**, where both variants in every pair come from the input.

```
Input variants (rsID, chr:pos, or chr:pos:ref:alt)
    ↓  DB lookup + window_bp
Genes overlapping input variants
    ↓  group membership (Pathway, GO, Disease, …)
Groups
    ↓  co-membership → Gene×Gene pairs
Gene×Gene pairs  [weight = # shared groups]
    ↓  cartesian of input variants per gene
Variant×Variant pairs  ← output
```

`group_support_count` is the biological weight: how many distinct groups connect the two genes.

See the explain guide: `biofilter/modules/report/reports_explain/report_variant_modeling.md`


### 1. Start Biofilter

In [ ]:
from biofilter import Biofilter

bf = Biofilter(debug_mode=False)
bf

### 2. Inspect report metadata

In [ ]:
report_name = 'variant_modeling'

print('name:', report_name)
print('\navailable columns:')
print(bf.report.available_columns(report_name))

print('\nexample_input:')
print(bf.report.example_input(report_name))

print('\nexplain:')
print(bf.report.explain(report_name))

### 3. Basic run — rsID inputs, Pathway grouping

Input: four variants from the APOE region and PCSK9.  
Grouping: Pathway (Reactome).  
Both variants in every output pair are from the input list.

In [ ]:
df = bf.report.run(
    'variant_modeling',
    input_data=[
        'rs429358',       # APOE ε4
        'rs7412',         # APOE ε2
        'rs2479409',      # PCSK9 promoter
        'rs11591147',     # PCSK9 R46L (loss-of-function)
    ],
    build=38,
    window_bp=0,
    group_entity_groups=['Pathway'],
)

print(f'Pairs: {len(df):,}')
df.head(20)

#### 3b. Top pairs by biological weight

In [ ]:
cols = [
    'variant_1_rsid', 'gene_1_name',
    'variant_2_rsid', 'gene_2_name',
    'group_support_count', 'group_support_names',
    'data_source_support_names',
]

df[cols].sort_values('group_support_count', ascending=False).head(20)

### 4. Mixed input — rsID, chr:pos, and chr:pos:ref:alt

The three formats can be mixed in the same list:

| Format | Example | Behavior |
|---|---|---|
| **rsID** | `rs429358` | dbSNP lookup |
| **chr:pos** | `chr19:44908684` | All alleles at the position (SNVs only) |
| **chr:pos:ref:alt** | `chr19:44908684:T:C` | Only the exact ref/alt variant (SNV or indel) |

Use `chr:pos:ref:alt` for credible-set / fine-mapping variants to avoid multiallelic ambiguity.

**Joining back to your source table.** Since BF4 4.1.4 the output carries two new columns,
`variant_1_input` and `variant_2_input`, that preserve the exact string you supplied. Use them
as the join key — no need to reparse `chr:pos:ref:alt` or look up rsID-to-coords after the fact.
Each variant also exposes `variant_*_ref` and `variant_*_alt` so multiallelic sites are
disambiguated.


In [ ]:
df_mixed = bf.report.run(
    'variant_modeling',
    input_data=[
        'rs429358',                # rsID
        'chr19:44908684',          # chr:pos (APOE region — all alleles at position)
        '2:21044574',              # bare chr:pos (APOB region)
        'chr19:44908684:T:C',      # chr:pos:ref:alt (exact APOE ε4 allele only)
    ],
    build=38,
    group_entity_groups=['Pathway'],
)

print(f'Pairs: {len(df_mixed):,}')

# Show the new 4.1.4 columns alongside the basics
preview_cols = [
    'variant_1_input', 'variant_1_rsid', 'variant_1_ref', 'variant_1_alt', 'gene_1_name',
    'variant_2_input', 'variant_2_rsid', 'variant_2_ref', 'variant_2_alt', 'gene_2_name',
    'group_support_count',
]
df_mixed[preview_cols].head(20)


#### 4b. Merging results back to a credible-set table

When the input came from a credible-set TSV (`locus / trait / SNP` columns, `SNP` in
`chr:pos:ref:alt` form), `variant_*_input` lets you merge the pair output directly back
to the source — no preprocessing, no rsID lookups, no allele parsing.

This is the canonical post-processing step for fine-mapping workflows.


In [ ]:
# Simulated credible-set table (same shape as multi_credible_sets_variants.tsv)
import pandas as pd

cs = pd.DataFrame({
    'locus': ['locus_apoe', 'locus_apoe', 'locus_apob'],
    'trait': ['LDL', 'LDL', 'LDL'],
    'SNP':   ['chr19:44908684:T:C', 'rs429358', '2:21044574'],
})

# Inner join on the input string — variant_1_input matches the SNP column verbatim
merged = cs.merge(
    df_mixed,
    left_on='SNP',
    right_on='variant_1_input',
    how='inner',
)

print(f'Merged rows: {len(merged):,}')
merged[[
    'locus', 'trait', 'SNP',
    'variant_1_input', 'variant_1_rsid', 'gene_1_name',
    'variant_2_input', 'variant_2_rsid', 'gene_2_name',
    'group_support_count',
]].head(20)


### 5. Multiple group types — Pathway + GO + Disease

Using multiple group types increases `group_support_count` when genes share more than one biological context.

In [ ]:
df_multi = bf.report.run(
    'variant_modeling',
    input_data=[
        'rs429358',
        'rs7412',
        'rs2479409',
        'rs11591147',
    ],
    build=38,
    group_entity_groups=['Pathway', 'GO', 'Disease'],
)

print(f'Pairs: {len(df_multi):,}')
df_multi[cols].sort_values('group_support_count', ascending=False).head(20)

#### 5b. group_support_count distribution

In [ ]:
import matplotlib.pyplot as plt

if not df_multi.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    df_multi['group_support_count'].value_counts().sort_index().plot(
        kind='bar', ax=ax, color='steelblue', edgecolor='white'
    )
    ax.set_xlabel('group_support_count (weight)')
    ax.set_ylabel('# variant pairs')
    ax.set_title('Biological weight distribution across variant pairs')
    plt.tight_layout()
    plt.show()

#### 5c. Gene pair heatmap (group_support_count)

In [ ]:
import pandas as pd

if not df_multi.empty:
    gene_pair_weight = (
        df_multi.groupby(['gene_1_name', 'gene_2_name'])['group_support_count']
        .max()
        .reset_index()
    )

    pivot = gene_pair_weight.pivot(index='gene_1_name', columns='gene_2_name', values='group_support_count').fillna(0)

    fig, ax = plt.subplots(figsize=(max(6, len(pivot.columns)), max(4, len(pivot))))
    im = ax.imshow(pivot.values, aspect='auto', cmap='Blues')
    plt.colorbar(im, ax=ax, label='group_support_count')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticklabels(pivot.index)
    ax.set_title('Gene pair biological weight (max group_support_count)')
    plt.tight_layout()
    plt.show()

    print('Gene pair summary:')
    display(gene_pair_weight.sort_values('group_support_count', ascending=False))

### 6. Restrict to a specific data source

Use `group_data_sources` to filter group membership to a single source (e.g., Reactome only).

In [ ]:
df_reactome = bf.report.run(
    'variant_modeling',
    input_data=['rs429358', 'rs7412', 'rs2479409', 'rs11591147'],
    group_entity_groups=['Pathway'],
    group_data_sources=['Reactome'],
)

print(f'Reactome-only pairs: {len(df_reactome):,}')
df_reactome[cols].head(20)

### 7. Window extension

`window_bp` extends gene boundaries when assigning variants to genes.
Useful when variants fall in regulatory regions near gene loci.

In [ ]:
results = {}
for window in [0, 5_000, 25_000]:
    df_w = bf.report.run(
        'variant_modeling',
        input_data=['rs429358', 'rs7412', 'rs2479409', 'rs11591147'],
        group_entity_groups=['Pathway'],
        window_bp=window,
    )
    results[window] = len(df_w)
    print(f'window_bp={window:>6,}  →  {len(df_w):,} pairs')

### 8. Input from file

Pass a path to a plain-text file (one rsID or chr:pos per line).

In [ ]:
from pathlib import Path

# Create a temporary input file for the tutorial
tmp_dir = Path('tmp/variant_modeling_tutorial')
tmp_dir.mkdir(parents=True, exist_ok=True)

input_file = tmp_dir / 'variants.txt'
input_file.write_text('rs429358\nrs7412\nrs2479409\nrs11591147\n')

df_file = bf.report.run(
    'variant_modeling',
    input_data=str(input_file),
    group_entity_groups=['Pathway'],
)

print(f'Pairs from file: {len(df_file):,}')
df_file[cols].head(10)

### 9. Safety check — max_pairs

The report estimates pair count before materialising. If the estimate exceeds `max_pairs` it aborts safely.

In [ ]:
df_safe = bf.report.run(
    'variant_modeling',
    input_data=['rs429358', 'rs7412', 'rs2479409', 'rs11591147'],
    group_entity_groups=['Pathway', 'GO', 'Disease'],
    max_pairs=5,   # intentionally low to trigger the check
)

if 'resolution_status' in df_safe.columns:
    print('Safety abort triggered:')
    print(df_safe[['resolution_status', 'estimated_pairs', 'max_pairs', 'suggestion']].to_string())
else:
    print(f'{len(df_safe):,} pairs — no abort')

### 10. Export results

In [ ]:
output_path = tmp_dir / 'variant_modeling_pairs.csv'
df_multi.to_csv(output_path, index=False)
print(f'Saved {len(df_multi):,} pairs → {output_path}')

### 11. Running on the UPenn LPC (Apptainer)

For cohort-scale runs (thousands of input variants × pathway/GO/Disease grouping), the **Penn LPC**
is usually the right place to execute this report. The Apptainer image bundles BF4 + PostgreSQL —
no local DB required.

> **Why LPC for `variant_modeling` specifically**
> - Pair generation scales as O(N²) on input — large lists hit the `max_pairs` cap fast and benefit from cluster RAM.
> - The group co-membership joins (pathways × genes × variants) are I/O-heavy and run faster with the DB co-located in the container.
> - Credible-set studies typically produce `chr:pos:ref:alt` lists in the 1k–10k range — local notebook connection latency adds up.

See also:
- [`lpc__quickstart.md`](lpc__quickstart.md) — minimal copy-paste recipe for first runs
- [`lpc__deploy.md`](lpc__deploy.md) — maintainer guide for installing / updating the LPC image and DB


#### 11a. Prepare the input file

Plain text, one variant per line. The `chr:pos:ref:alt` form is preferred for credible sets:

```bash
module load apptainer
export WORKSPACE=/project/<your-project>/bf4_runs

cat > "$WORKSPACE/cs_variants.txt" <<'EOF'
1:6203732:A:G
1:46108752:C:T
2:71307487:T:A
chr19:44908684:T:C
EOF
```

The single-quoted `<<'EOF'` heredoc preserves the colons literally. For a TSV in
`locus / trait / SNP` format (typical credible-set output), pipe the `SNP` column:

```bash
awk -F'\t' 'NR>1 {print $3}' credible_sets.tsv | sort -u > "$WORKSPACE/cs_variants.txt"
```


#### 11b. Prepare the params file (avoids shell-quote breakage)

`variant_modeling` takes list parameters (`group_entity_groups`, `group_data_sources`).
**Passing them inline with `--param 'KEY=["Pathway"]'` fails** — the shell inside the container
strips the inner double quotes and you get errors like `No valid group_entity_groups found for ['[pathway]']`.

The robust pattern is a JSON file referenced via `--params-file`:

```bash
cat > "$WORKSPACE/vm_params.json" <<'EOF'
{
  "group_entity_groups": ["Pathway"],
  "window_bp": 0,
  "max_pairs": 1000000
}
EOF
```

For multi-group / multi-source runs:

```bash
cat > "$WORKSPACE/vm_params.json" <<'EOF'
{
  "group_entity_groups": ["Pathway", "GO", "Disease"],
  "group_data_sources": ["Reactome"],
  "window_bp": 5000,
  "max_pairs": 5000000
}
EOF
```

Sanity-check the JSON before running it through the container:

```bash
python3 -c "import json; print(json.load(open('$WORKSPACE/vm_params.json')))"
```


#### 11c. Run the report

Same boilerplate as `annotation_master_variant` — only `--name` and the file references change.
The temp dir + bind mounts give PostgreSQL inside the container its scratch space.

```bash
TMP=$(mktemp -d) && mkdir -p "$TMP/tmp" "$TMP/pg-run" && \
apptainer run --writable-tmpfs --pwd /tmp \
  --bind /project/hall_shared/biofilter/databases/20260514/pgdata:/var/lib/postgresql/data \
  --bind "$TMP/tmp:/tmp" \
  --bind "$TMP/pg-run:/var/run/postgresql" \
  --bind "$WORKSPACE:/workspace" \
  /project/hall_shared/biofilter/images/bf4-hpc-4.1.2.sif \
  biofilter report run \
    --name variant_modeling \
    --input-file /workspace/cs_variants.txt \
    --params-file /workspace/vm_params.json \
    --output /workspace/variant_modeling_pairs.csv && \
rm -rf "$TMP"
```

Result: `$WORKSPACE/variant_modeling_pairs.csv`.

> **Safety check first.** Start with `"max_pairs": 1000000` — if the estimator aborts, the CSV
> will contain a single row with `resolution_status`, `estimated_pairs`, `max_pairs`, and a
> `suggestion` column telling you how to tighten the filter. Re-tune (`group_data_sources`,
> stricter `group_entity_groups`, smaller `window_bp`) before raising the cap.


### 12. Real cohort template

Replace the input list with your study variants and adjust group filters.


In [ ]:
# Option A: explicit list (rsID, chr:pos, or chr:pos:ref:alt)
my_variants = [
    'rs429358',
    # ... add your variants
]

# Option B: load from file (one entry per line; mixed formats supported)
# my_variants = '/path/to/variants.txt'

# For credible-set / fine-mapping cohorts, prefer chr:pos:ref:alt to avoid
# multiallelic ambiguity at SNP positions:
#   ['1:6203732:A:G', 'chr19:44908684:T:C', ...]

# df_cohort = bf.report.run(
#     'variant_modeling',
#     input_data=my_variants,
#     build=38,
#     window_bp=0,
#     group_entity_groups=['Pathway', 'GO'],
#     group_data_sources=['Reactome'],
#     max_pairs=1_000_000,
# )

# print(f'Cohort pairs: {len(df_cohort):,}')
# df_cohort.to_csv('outputs/variant_modeling_cohort.csv', index=False)
# df_cohort.head(20)
